# Censored Demand Analysis

**Goal**: Analyze the impact of stockouts or observed demand and identify high-risk stores

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

# Set style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries Loaded")

Libraries Loaded


## 1. Load the data

In [3]:
# Load the dataset
df = pd.read_parquet('../data/generated/sales_data.parquet')

df['date'] = pd.to_datetime(df['date'])

In [4]:
print("Dataset shape:", df.shape)
print(f"Date range: {df['date'].min()} - {df['date'].max()}")
print("\nKey Columns:")
print("* demand: True Demand (units)")
print("* sales_qty: Actual Sales (capped by inventory)")
print("* inventory: Stock available")
print("* stockout_flag: 1 if demand > inventory")
print("* lost_sales: demand - sales_qty (if demand > inventory)")

display(df[['date', 'store', 'category', 'demand', 'sales_qty', 'inventory', 'stockout_flag', 'lost_sales']].head(10))

Dataset shape: (2883231, 55)
Date range: 2016-01-01 00:00:00 - 2026-08-26 00:00:00

Key Columns:
* demand: True Demand (units)
* sales_qty: Actual Sales (capped by inventory)
* inventory: Stock available
* stockout_flag: 1 if demand > inventory
* lost_sales: demand - sales_qty (if demand > inventory)


,date,store,category,demand,sales_qty,inventory,stockout_flag,lost_sales
0,2016-01-01,Bandra Hub,Electronics,11,11,12,0,0
1,2016-01-01,Bandra Hub,Electronics,33,33,39,0,0
2,2016-01-01,Bandra Hub,Electronics,34,34,49,0,0
3,2016-01-01,Bandra Hub,Electronics,23,23,25,0,0
4,2016-01-01,Bandra Hub,Electronics,15,15,18,0,0
5,2016-01-01,Bandra Hub,Clothing,45,45,50,0,0
6,2016-01-01,Bandra Hub,Clothing,49,49,57,0,0
7,2016-01-01,Bandra Hub,Clothing,60,60,89,0,0
8,2016-01-01,Bandra Hub,Clothing,24,24,33,0,0
9,2016-01-01,Bandra Hub,Clothing,14,14,15,0,0


## 2. Overview Statistics: Censoring Problem

In [5]:
print("CENSORING PROBLEM OVERVIEW")
print("="*70)

# GLobal censoring rate
total_rows = len(df)
censored_rows = df['stockout_flag'].sum()
censored_rate = censored_rows / total_rows

print(f"\n Total Observations: {total_rows:,}")
print(f"Censored (Stockout): {censored_rows:,} ({censored_rate*100:.2f}%)")
print(f"Uncensored: {total_rows - censored_rows:,} ({(1-censored_rate)* 100:.2f}%)")

CENSORING PROBLEM OVERVIEW

 Total Observations: 2,883,231
Censored (Stockout): 822,530 (28.53%)
Uncensored: 2,060,701 (71.47%)


In [6]:
# Lost Sales Impact
total_lost_sales = df['lost_sales'].sum()
total_actual_sales = df['sales_qty'].sum()
true_demand_sum = df['demand'].sum()

print("\nREVENUE IMPACT")
print(f"Total Actual Sales (qty): {total_actual_sales:,}")
print(f"Total Lost Sales (qty): {total_lost_sales:,}")
print(f"True Demand(qty): {true_demand_sum:,}")
print(f"Lost as % of True Demand: {(total_lost_sales / true_demand_sum) * 100:.2f}%")


REVENUE IMPACT
Total Actual Sales (qty): 165,840,506
Total Lost Sales (qty): 5,315,734
True Demand(qty): 171,156,240
Lost as % of True Demand: 3.11%


In [11]:
# Estimated lost revenue
avg_price = df['final_price'].mean()
estimated_lost_revenue = total_lost_sales * avg_price
print(f"\nEstimated Lost Revenue: Rs.{estimated_lost_revenue:,.0f}")


Estimated Lost Revenue: Rs.71,304,646,924


In [10]:
# Distribution of Censoring
print(f"\nCENSORING DISTRIBUTION:")
censoring_dist = pd.cut(
    df.groupby('store_id')['stockout_flag'].mean(),
    bins = [0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels = ['0-10%', '10-20%', '20-30%', '30-50%', '50-100%']
)
print(censoring_dist.value_counts().sort_index())


CENSORING DISTRIBUTION:
stockout_flag
0-10%       0
10-20%      0
20-30%     37
30-50%      0
50-100%     0
Name: count, dtype: int64


## 3. Estimated True Demand

In [12]:
# Calculate estimated true demand
# For uncensored: use observed demand
# For censored: add back lost sales to obsereved

df['estimated_true_demand'] = df['demand'].copy()
# Demand already includes lost_sales for censored cases in synthetic data, so we don't need to add lost_sales again.

print("DEMAND COMPARISON")
print("="*70)

comparison = df[['demand', 'sales_qty', 'inventory', 'lost_sales', 'estimated_true_demand']].describe()
print(comparison)

DEMAND COMPARISON
             demand     sales_qty     inventory    lost_sales  \
count  2.883231e+06  2.883231e+06  2.883231e+06  2.883231e+06   
mean   5.936265e+01  5.751898e+01  6.776669e+01  1.843673e+00   
std    6.380661e+01  6.217109e+01  7.544788e+01  5.237837e+00   
min    1.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
25%    1.800000e+01  1.800000e+01  2.000000e+01  0.000000e+00   
50%    3.600000e+01  3.500000e+01  4.100000e+01  0.000000e+00   
75%    7.700000e+01  7.500000e+01  8.800000e+01  1.000000e+00   
max    9.470000e+02  9.470000e+02  1.331000e+03  1.410000e+02   

       estimated_true_demand  
count           2.883231e+06  
mean            5.936265e+01  
std             6.380661e+01  
min             1.000000e+00  
25%             1.800000e+01  
50%             3.600000e+01  
75%             7.700000e+01  
max             9.470000e+02  


In [14]:
print("\n🔍 Examples of Censored vs Uncensored Observations:")
print("\nCENSORED (Stockout):")
display(df[df['stockout_flag'] == 1][['date', 'store', 'category', 'demand', 'sales_qty', 'inventory', 'lost_sales']].head(5))

print("\nUNCENSORED (No Stockout):")
display(df[df['stockout_flag'] == 0][['date', 'store', 'category', 'demand', 'sales_qty', 'inventory', 'lost_sales']].head(5))


🔍 Examples of Censored vs Uncensored Observations:

CENSORED (Stockout):


,date,store,category,demand,sales_qty,inventory,lost_sales
16,2016-01-01,Bandra Hub,Grocery,69,61,61,8
20,2016-01-01,Borivali Plaza,Electronics,35,32,32,3
30,2016-01-01,Borivali Plaza,Clothing,53,50,50,3
34,2016-01-01,Borivali Plaza,Grocery,53,49,49,4
37,2016-01-01,Borivali Plaza,Grocery,94,78,78,16



UNCENSORED (No Stockout):


,date,store,category,demand,sales_qty,inventory,lost_sales
0,2016-01-01,Bandra Hub,Electronics,11,11,12,0
1,2016-01-01,Bandra Hub,Electronics,33,33,39,0
2,2016-01-01,Bandra Hub,Electronics,34,34,49,0
3,2016-01-01,Bandra Hub,Electronics,23,23,25,0
4,2016-01-01,Bandra Hub,Electronics,15,15,18,0
